# 04 — Sub-Issue Classification

**Project:** Customer Complaint Root-Cause Classifier  
**Module:** 3 — Granular Classification of Complaint Sub-Issues  
**Target:** `sub_issue` (38 categories)  
**Input:** `complaint_narrative`  

### Objectives:
1. Load and prepare complaints data for granular sub-issue modeling.
2. Analyze the **hierarchical relationship** between the 21 main issues and 38 sub-issues.
3. Apply **Group-Aware Splitting** (`GroupShuffleSplit` on narrative MD5 hash) to eliminate synthetic duplicate text leakage.
4. Train and benchmark 3 algorithms:
   - **TF-IDF + Multinomial Logistic Regression** (balanced class weights)
   - **TF-IDF + Linear Support Vector Classifier (LinearSVC)**
   - **TF-IDF + Multinomial Naive Bayes**
5. Evaluate using Accuracy, Macro F1, Weighted F1, and Precision/Recall.
6. Demonstrate **Hierarchical Classification** (Issue -> Sub-Issue taxonomy consistency).
7. Save champion model artifacts to `models/sub_issue_classifier.joblib`.

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path(os.path.abspath("")).resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

from config.settings import (
    PROJECT_ROOT, MODELS_DIR, RANDOM_SEED, TEST_SIZE, TFIDF_MAX_FEATURES
)
from config.labels import SUB_ISSUE_LABELS, ISSUE_TO_SUB_ISSUES, SUB_ISSUE_TO_ISSUE
from src.data.load_data import load_complaints
from src.data.clean_data import clean_complaints
from src.utils.helpers import generate_narrative_hash, get_memory_usage
from src.classification.evaluate import evaluate_model, plot_confusion_matrix, compare_models

print(f"Project root: {PROJECT_ROOT}")
print(f"Memory usage: {get_memory_usage():.2f} MB")

## 2. Load and Inspect Sub-Issue Distribution

In [ ]:
print("Loading and cleaning complaint dataset...")
raw_df = load_complaints()
df = clean_complaints(raw_df)

sub_counts = df["sub_issue"].value_counts()
print(f"Total complaints: {len(df):,}")
print(f"Unique sub-issues: {len(sub_counts)}")
print("\nTop 10 Sub-Issues:")
print(sub_counts.head(10).to_string())

## 3. Hierarchical Structure: Issues vs Sub-Issues

In [ ]:
# Sub-issues count per issue
subs_per_issue = {issue: len(subs) for issue, subs in ISSUE_TO_SUB_ISSUES.items()}
subs_df = pd.DataFrame(list(subs_per_issue.items()), columns=["Issue", "Sub-Issue Count"]).sort_values("Sub-Issue Count", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=subs_df, x="Sub-Issue Count", y="Issue", palette="crest")
plt.title("Number of Sub-Issues per Parent Issue", fontsize=13, fontweight="bold")
plt.xlabel("Count of Distinct Sub-Issues")
plt.tight_layout()
plt.show()

print(f"Issues with exactly 1 sub-issue (deterministic): {(subs_df['Sub-Issue Count'] == 1).sum()}")
print(f"Issues with >1 sub-issues: {(subs_df['Sub-Issue Count'] > 1).sum()}")

## 4. Group-Aware Split (Prevent Duplicate Narrative Leakage)

In [ ]:
X = df["complaint_narrative"]
y_raw = df["sub_issue"]

# MD5 narrative hash for group integrity
groups = X.apply(generate_narrative_hash)

# Label encoding for 38 classes
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
classes = label_encoder.classes_

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train set: {len(X_train):,} ({len(X_train)/len(df):.1%})")
print(f"Test set:  {len(X_test):,} ({len(X_test)/len(df):.1%})")

# Check group isolation
overlap = set(groups.iloc[train_idx]).intersection(set(groups.iloc[test_idx]))
assert len(overlap) == 0, "Group leakage detected!"
print("Group isolation verified: Zero overlap between train and test narratives.")

## 5. TF-IDF Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.95,
)

print("Fitting TF-IDF on sub-issue narratives...")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"X_train shape: {X_train_vec.shape}")
print(f"X_test shape:  {X_test_vec.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")

## 6. Model Training & Comparison

In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
    "Linear SVM": LinearSVC(
        max_iter=2000,
        class_weight="balanced",
        dual="auto",
        random_state=RANDOM_SEED,
    ),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
}

evaluations = []
trained_models = {}
predictions = {}

for name, model in candidate_models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train_vec, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_vec)
    predictions[name] = y_pred
    
    metrics = evaluate_model(y_test, y_pred, labels=classes, model_name=name)
    evaluations.append(metrics)
    print(f"{name} -> Accuracy: {metrics['accuracy']:.4f} | Macro F1: {metrics['macro_f1']:.4f}")

## 7. Sub-Issue Model Benchmark

In [ ]:
comparison_df = compare_models(evaluations)
print("=== SUB-ISSUE CLASSIFICATION BENCHMARK ===")
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print(f"Champion Sub-Issue Classifier: {best_model_name}")

# Plot confusion matrix
fig = plot_confusion_matrix(
    y_test,
    predictions[best_model_name],
    labels=classes,
    model_name=best_model_name,
    figsize=(18, 14)
)
plt.title(f"Normalized Confusion Matrix: {best_model_name} (38 Sub-Issues)", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Save Model Artifacts

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

sub_model_file = MODELS_DIR / "sub_issue_classifier.joblib"
sub_vec_file = MODELS_DIR / "tfidf_sub_issue_vectorizer.joblib"
sub_le_file = MODELS_DIR / "sub_issue_label_encoder.joblib"

joblib.dump(best_model, sub_model_file)
joblib.dump(vectorizer, sub_vec_file)
joblib.dump(label_encoder, sub_le_file)

print(f"Saved sub-issue classifier -> {sub_model_file}")
print(f"Saved TF-IDF vectorizer    -> {sub_vec_file}")
print(f"Saved label encoder        -> {sub_le_file}")

## 9. Hierarchical Classification Inference Test

In [ ]:
from src.classification.predict import predict_hierarchical

test_complaints = [
    "I was charged twice for a single transaction on my debit card statement.",
    "My account was closed by the bank without any written notice or explanation.",
    "Debt collector has been calling repeatedly at work despite my written request.",
    "An unexpected overdraft fee was assessed even though my balance was positive.",
    "The interest rate on my loan jumped unexpectedly without any notice."
]

print("=== HIERARCHICAL PREDICTIONS (ISSUE -> SUB-ISSUE) ===\n")
for text in test_complaints:
    h_res = predict_hierarchical(text)
    print(f"Complaint:  {text}")
    print(f"Issue:      {h_res['issue']} (Confidence: {h_res['issue_confidence']:.2%})")
    print(f"Sub-Issue:  {h_res['sub_issue']} (Confidence: {h_res['sub_issue_confidence']:.2%})")
    print(f"Taxonomy Consistent: {h_res['is_hierarchically_consistent']}")
    print("-" * 72)